# Exploratory Analysis — Disaster Tweets

This notebook performs a small, reproducible first look at the Kaggle **Natural Language Processing with Disaster Tweets** training set. It does not fit models or inspect the competition test labels.

Download `train.csv` from [Kaggle](https://www.kaggle.com/competitions/nlp-getting-started/data) and place it in `data/raw/` before running the notebook.

In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt

DATA_PATH = Path("../data/raw/train.csv") if Path("../data/raw/train.csv").exists() else Path("data/raw/train.csv")
if not DATA_PATH.exists():
    raise FileNotFoundError("Place Kaggle's train.csv in data/raw/ before running this notebook.")

df = pd.read_csv(DATA_PATH)
required = {"id", "text", "target"}
missing = required.difference(df.columns)
if missing:
    raise ValueError(f"Missing required columns: {sorted(missing)}")

df["text"] = df["text"].fillna("").astype(str)
df.head()

## 1. Dataset overview

Check the data shape, column types, missing values, duplicate identifiers, and class balance. The `keyword` and `location` fields are not used by the baseline/deep-learning experiment in this repository; the main experiment focuses on tweet text.

In [ ]:
print(f"Rows: {len(df):,} | Columns: {len(df.columns)}")
display(df.dtypes.to_frame("dtype"))
display(df.isna().sum().to_frame("missing_values"))
print("Duplicate IDs:", df["id"].duplicated().sum())
display(df["target"].value_counts().sort_index().rename(index={0: "Not disaster", 1: "Disaster"}).to_frame("count"))

In [ ]:
class_counts = df["target"].value_counts().sort_index()
fig, ax = plt.subplots(figsize=(6, 4))
ax.bar(["Not disaster (0)", "Disaster (1)"], class_counts.values)
ax.set_title("Target class distribution")
ax.set_ylabel("Number of tweets")
ax.grid(axis="y", alpha=0.25)
plt.show()

## 2. Tweet length

Tweet length can be measured in characters or whitespace-separated tokens. This helps choose a reasonable sequence length for a neural model. The training script defaults to 64 tokens and truncates longer examples.

In [ ]:
df["character_count"] = df["text"].str.len()
df["word_count"] = df["text"].str.split().str.len()
display(df[["character_count", "word_count"]].describe(percentiles=[0.5, 0.9, 0.95, 0.99]).T)

fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(df["word_count"], bins=40)
ax.set_title("Distribution of tweet word counts")
ax.set_xlabel("Whitespace-separated words")
ax.set_ylabel("Frequency")
ax.grid(alpha=0.25)
plt.show()

## 3. Representative examples

Read a few examples from each class. The same word or phrase may be literal, figurative, sarcastic, or unrelated to a real event. This ambiguity is one reason a text classifier can make confident mistakes.

In [ ]:
for label, title in [(0, "Not disaster"), (1, "Disaster")]:
    print(f"\n--- {title} examples ---")
    examples = df.loc[df["target"] == label, "text"].sample(
        n=min(5, int((df["target"] == label).sum())), random_state=42
    )
    for example in examples:
        print("•", example)

## 4. What this analysis tells us

- Confirm whether the two classes are similarly represented before selecting metrics.
- Inspect very short and very long tweets; sequence truncation may remove useful context.
- Preserve hashtags and most punctuation during normalization because they can carry meaning.
- Use a stratified validation split and fit vocabularies/vectorizers on training texts only.

Continue with model training from the repository root:

```bash
python -m disaster_tweets.train --data data/raw/train.csv
```